# 05 · 组合构建

> 本 notebook 是《量化研究入门学习资料》第 X 章的可运行配套。
> 数据源：`data/csv/`。运行前请先执行 `python scripts/generate_data.py` 生成数据。

## 目标
用等权合成因子构建 Top20 等权组合，重算净值/换手/成本敏感性，对照 `portfolio.json`。

In [ ]:
import pandas as pd, numpy as np, json
factors = pd.read_csv("data/csv/factors.csv", parse_dates=["date"])
FACTORS = ["EP", "SIZE", "MOM60", "REV5", "VOL20", "TURN", "ROE", "GROW"]
w = np.full(8, 1 / 8)

# ---- 逐期 Top20 等权组合 ----
nav, bench, ls, turnover = [1.0], [1.0], [1.0], []
prev = None
for _, g in factors.groupby("date"):
    z = g[[f"z_{f}" for f in FACTORS]].fillna(0.0).values @ w
    y = g["next_return"].values
    if np.isnan(y).all():
        nav.append(nav[-1]); bench.append(bench[-1]); ls.append(ls[-1]); turnover.append(0.0); continue
    order = np.argsort(-z)
    order = order[y[order] == y[order]]          # 剔除退市/无标签
    top, bot = order[:20], order[-20:]
    rp, rb = y[top].mean(), np.nanmean(y)
    nav.append(nav[-1] * (1 + rp)); bench.append(bench[-1] * (1 + rb))
    ls.append(ls[-1] * (1 + rp - y[bot].mean()))
    to = len(set(top) - (prev or set())) * 2 / 20
    turnover.append(to if prev is not None else 0.0)
    prev = set(top)

# ---- 成本敏感性（单边 bps）----
nav_costs = {"0": nav[1:]}
for name, c in [("5", 5e-4), ("10", 1e-3), ("20", 2e-3)]:
    v, seq = 1.0, []
    for to in turnover:
        v *= (1 - to * c); seq.append(v)
    nav_costs[name] = seq

print("组合期末净值:", round(nav[-1], 3), "| 基准:", round(bench[-1], 3),
      "| 平均换手:", f"{np.mean(turnover):.0%}")

In [ ]:
# ---- 对照 portfolio.json ----
ref = json.load(open("data/portfolio.json", encoding="utf-8"))
ok_nav = np.allclose(nav[1:], ref["nav"], atol=1e-5)
ok_turn = np.allclose(turnover, ref["turnover"], atol=1e-3)
ok_cost = np.allclose(nav_costs["20"], ref["nav_costs"]["20"], atol=1e-5)
print("净值对照:", "PASS" if ok_nav else "FAIL")
print("换手对照:", "PASS" if ok_turn else "FAIL")
print("成本敏感性对照:", "PASS" if ok_cost else "FAIL")
assert ok_nav and ok_turn and ok_cost